# Proyecto — «Puente»: ML causal en producción local

**Semana 4 · construcción guiada en clase + tarea individual sobre `real/ihdp`.**

En este proyecto construyes, pieza por pieza, un servicio causal local completo: un estimador con incertidumbre, una tarjeta de modelo, un registro de versiones con hash y alias `champion`, un servicio FastAPI que se abstiene fuera del soporte, y una puerta causal `pasa / alerta / falla` sobre ventanas de producción. Es el esqueleto del proyecto final del curso, en miniatura.

## El mapa

| Concepto del libro | Cap. | Se materializa en |
|---|---|---|
| El estimando se declara antes de mirar | 1 | `pregunta.md` (léela primero) |
| Cinco acciones; abstención de primera clase | 2 | `monitor.py`, `servicio_core.py` |
| Contrato C-01 (roles) | 3 | `datos.py` |
| Contrato C-09 (hashes, versión, champion) | 4 | `registro.py` |
| Contrato C-10 (splits sembrados, la pared) | 5 | `datos.py` |
| Validez temporal: la trampa | 6 | `score_post` en `estimacion.py` |
| C-02: SMD / PSI / ESS con umbrales calibrados | 7 | `estimacion.py`, `monitor.py` |
| Benchmark como producto (C-11) | 8 | todo corre sobre un escenario declarado |
| (puente) Lo que DML le exige al dato | 9 | `aipw_crossfit` en `estimacion.py` |

## Cómo se trabaja

1. **El código se escribe en los archivos `.py`**, no en el notebook: cada uno trae TODOs numerados con pistas (TODO-1 … TODO-22). Este notebook es el conductor: explica, ejecuta los checkpoints y deja constancia de tus resultados.
2. Cada bloque de la clase termina con un `make <objetivo>` que debe salir en verde con los números esperados.
3. La **tarea** (sección 7) reemplaza el escenario `real/jobs` por `real/ihdp` y es lo que se entrega (ver `ENTREGA.md`).


## 0. Preparación

Entorno del curso activado (`python -m pip install -r requirements.txt` si falta algo).
El estado de arranque correcto: `make verificar` **falla** con `TODO-1` — así se empieza.

> Si tu máquina no tiene `make`, los objetivos son atajos de un comando: `make verificar` = `python3 datos.py`, `make estimar` = `python3 estimacion.py`, `make registrar` = `python3 registro.py`, `make ventanas` = `python3 monitor.py`, `make humo` = `python3 humo.py`.

In [ ]:
# Entorno: debe imprimir las versiones y los archivos del proyecto
import pathlib, sys
print("python:", sys.version.split()[0])
try:
    import numpy, scipy
    print("numpy", numpy.__version__, "| scipy", scipy.__version__)
except ImportError as e:
    print("FALTA:", e, "-> pip install -r requirements.txt")

esperados = ["pregunta.md", "datos.py", "estimacion.py", "registro.py",
             "monitor.py", "servicio_core.py", "servicio.py", "humo.py", "Makefile"]
presentes = [f.name for f in pathlib.Path(".").glob("*")]
print("archivos:", ", ".join(f for f in esperados if f in presentes))
assert len([f for f in esperados if f in presentes]) == len(esperados)


In [ ]:
# Checkpoint de arranque: DEBE fallar con TODO-1 (estado inicial correcto)
!make verificar


Si la celda anterior no falló con `TODO-1`, algo viene ya resuelto: pide una copia limpia de este proyecto.

## 1. La pregunta antes que el código (Cap. 1)

Abre [`pregunta.md`](pregunta.md) y léela completa: pregunta causal, estimando, población, supuestos con su clase (`observable` / `no_observable`), y qué NO afirma el prototipo.

> **Para tu entrega:** en la sección 7 escribirás tu propia pregunta para `real/ihdp`, con el mismo formato. Ve anotando qué estimando te interesa y qué supuestos declararás.

## 2. El contrato de roles y la pared (Caps. 3 y 5)

Completas **TODO-1, TODO-2 y TODO-3** en `datos.py`:

- `TODO-1` — `EsquemaCausal.validar`: tres rechazos que *nombran al culpable* (columna ausente, doble rol, tratamiento constante).
- `TODO-2` — `verificar_splits`: recomputar el SHA-256 de las tres particiones contra lo que declaró el productor en `splits.json`.
- `TODO-3` — `comprobar_pared`: la verdad del escenario (`truth_*.csv`) no existe en el paquete; ese `FileNotFoundError` **es** la pared.

In [ ]:
# Checkpoint 1: los tres hash OK y la pared
!make verificar


**Salida esperada** (escenario `real/jobs`):

```text
esquema valida contra observed_train: OK (n = 1033, tratados = 262)
  train        hash OK  (1033 filas, semilla 20240701)
  validation   hash OK  (344 filas, semilla 20240701)
  test         hash OK  (345 filas, semilla 20240701)
  pared OK: truth_train.csv no existe en el paquete (reservado)
```

In [ ]:
# Provoca el fallo a propósito: el origen renombró una columna
import sys; sys.path.insert(0, ".")
from datos import EsquemaCausal
esquema = EsquemaCausal()
try:
    esquema.validar(["treatment", "re78", "age", "black"], [0, 1, 0, 1])  # sin 'educ'
except ValueError as e:
    print("rechazado como corresponde:", e)


## 3. El efecto con incertidumbre (Caps. 1, 6, 7 — y el puente al 9)

Completas **TODO-4 … TODO-8, TODO-21 y TODO-22** en `estimacion.py`:

- `TODO-4` — `ate_ols`: la referencia ingenua (OLS con controles + IC).
- `TODO-21` — **`aipw_crossfit` — la pieza del día**: el ATE doblemente robusto con cross-fitting K=5 — outcome por brazo y propensity ajustados *fuera* del fold que predicen, y el score AIPW por unidad (la influence function).
- `TODO-22` — `estratos_nodegree`: la media de scores por estrato con su IC, con el corte **pre-declarado** en la tarjeta.
- `TODO-8` — la trampa: una columna calculada con el resultado («score de éxito del programa»).

In [ ]:
# Checkpoint 2: el efecto, el AIPW y los estratos
!make estimar


**Salida esperada** (`real/jobs`):

```text
diferencia cruda E[Y|D=1]-E[Y|D=0] = 1600.8 USD
ATE OLS (8 controles) = 1596.0 | SE = 301.6 | IC95 = [1004.8, 2187.3]
AIPW cross-fitting K=5 = 1638.6 | SE = 325.0 | IC95 = [1001.6, 2275.7]
trampa score_post      = 172.4  <- el coeficiente se vació
estrato nodegree=1 (sin título, n=568): efecto 1060.1 IC95 [233.1, 1887.2]
estrato nodegree=0 (con título, n=465): efecto 2345.2 IC95 [1356.9, 3333.4]
ESS = 729 (ESS/n = 0.71) | rango e = [0.092, 0.530] | solapamiento = 100%
SMD máx = 0.19
```

**Cómo se lee:**

- **OLS y AIPW — dos estimadores distintos, misma conclusión** (1 596 vs 1 638): eso compra la doble robustez. Y ningún nuisance vio a la unidad que predice: la exigencia número uno del Cap. 9, en 15 líneas.
- **Los estratos sugieren** que el programa rinde más en quienes tienen título (2 345 vs 1 060), pero los intervalos **se solapan**: heterogeneidad sugerida, no concluyente. El servicio lo dirá con esa letra.
- **La trampa** vacía el coeficiente (172) sin lanzar ningún error: la fuga no penaliza el ajuste predictivo: al contrario, lo mejora. Por eso ninguna métrica de ajuste la detecta.

In [ ]:
%matplotlib inline
# Los estratos, en un gráfico: intervalos que se solapan
import json
import matplotlib.pyplot as plt

try:
    r = json.load(open("artifacts/estimacion.json"))
except FileNotFoundError:
    print("Corre antes: make estimar (con los TODO del bloque de estimación resueltos)")
    r = None

if r is not None:
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    labels, xs, errs = [], [], []
    for v, h in r["estratos"]["hojas"].items():
        labels.append("nodegree=" + str(v) + "\n(" + h["etiqueta"] + ", n=" + str(h["n"]) + ")")
        xs.append(h["efecto"])
        errs.append(1.96 * h["se"])
    ax.errorbar(xs, range(len(xs)), xerr=errs, fmt="o", capsize=4, color="#0E7490")
    ax.axvline(r["ate_aipw"], color="#A61E2B", ls="--", lw=1,
               label="ATE (AIPW) = {:.0f}".format(r["ate_aipw"]))
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("efecto del programa (USD de 1978)")
    ax.set_title("Efecto por estrato: heterogeneidad sugerida, intervalos que se solapan")
    ax.legend(); fig.tight_layout()
    fig  # última expresión: el notebook renderiza el PNG del gráfico


## 4. Tarjeta, registro y champion (Cap. 4)

Completas **TODO-9 … TODO-13** en `registro.py`: la tarjeta (con umbrales calibrados y supuestos clasificados), la versión inmutable `vNNNN` con manifiesto de hashes, la integridad que recomputa, y la promoción atómica de `champion` (que nunca promueve una puerta `falla`).

In [ ]:
# Checkpoint 3: registro + mutación detectada + champion
!make registrar


**Salida esperada:**

```text
registrada v0001 | integridad: OK
mutación detectada: SÍ (hash distinto del manifiesto -> bloqueado)
champion -> v0001
champion resuelve a: v0001
```

La mutación del ejemplo inyecta **un solo byte** en una copia de `estimacion.json`: el hash deja de coincidir con el manifiesto y la promoción queda bloqueada. La afirmación publicada se refiere a un sujeto con hash.

## 5. Ventanas y puerta causal (Caps. 2 y 7)

Completas **TODO-14 … TODO-18** en `monitor.py`: el PSI, las tres ventanas de producción (sana, deriva, positividad) y la puerta que decide `pasa / alerta / falla` contra los umbrales **declarados en la tarjeta**.

In [ ]:
# Checkpoint 4: las tres ventanas
!make ventanas


**Salida esperada:**

```text
[sana        ] estado = pasa   | acción: continuar
[deriva      ] estado = alerta | acción: alertar (revisar antes de reentrenar)
              - deriva: PSI de age = 1.35 (umbral 0.25)
[positividad ] estado = falla  | acción: abstenerse (bloquear la promoción)
              - solapamiento roto: 0% de unidades con e en [0.05, 0.95] (mínimo 50%)
```

**Responde en la celda siguiente** (2–4 líneas por pregunta):

1. En la ventana `positividad`, el ESS/n da 1.0 — ¿por qué esa cifra *no* es una buena señal, y cuál es la señal que sí rompe la puerta?
2. En la ventana `deriva` el SMD queda en 0.19 — ¿por qué la puerta no se alerta por balance, y qué pregunta distinta responde el PSI?
3. Con la señal de `deriva` en la mano: ¿cuál de las cinco acciones corresponde, y qué NO puedes afirmar sobre la causa del cambio?

In [ ]:
# Tus respuestas del bloque de ventanas
respuesta_v1 = """
...
"""
respuesta_v2 = """
...
"""
respuesta_v3 = """
...
"""


## 6. El servicio: abstención y efecto del estrato (Cap. 2 y Parte IV)

Completas **TODO-19 y TODO-20** en `servicio_core.py`: el bundle champion (el servicio **no** lee CSV — el bundle viaja, los datos no) y la consulta con abstención fuera del soporte **y efecto del estrato** dentro.

In [ ]:
# Checkpoint 5: el humo del servicio
!make humo


**Salida esperada:**

```text
/salud: 200 {'estado': 'ok', 'version': 'v0001', ...}
/efecto: 200 efecto=1638.6 ic95=[1001.6, 2275.7] version=v0001
/consulta dentro: 200 estado=respuesta efecto=1060.1
/consulta fuera (age=99): 200 estado=abstencion motivo=age=99 fuera de [18.0, 54.0] (sin número)
determinismo: dos llamadas idénticas -> iguales
HUMO: OK
```

Fíjate: la abstención devuelve HTTP **200** — es una respuesta de primera clase, no un error. Y `/efecto` y `/consulta` responden **estimandos distintos** (el poblacional contra el del estrato), cada uno declarando qué cubre su intervalo.

Para levantar el servicio de verdad (opcional, en una terminal):

```bash
make servir        # y en otra terminal:
curl -s localhost:8000/efecto | python -m json.tool
curl -s -X POST localhost:8000/consulta -H "Content-Type: application/json" -d '{"age": 99, "educ": 10, "black": 1, "hispan": 0, "married": 0, "nodegree": 1, "re74": 2000, "re75": 1500}' | python -m json.tool
```

## 7. TAREA — El mismo puente sobre `real/ihdp` (lo que se entrega)

El escenario `real/ihdp` (Hill 2011, sintético): 747 unidades, **25 covariables**, tratamiento `treatment`, resultado `outcome` (puntaje cognitivo a los 3,5 años), ATE de referencia ~4,0 según la literatura, y **efectos heterogéneos por diseño**. Las particiones son 448 / 149 / 150.

Tu trabajo: **adaptar el prototipo completo al nuevo escenario y reportar tus números.** Nadie te da el esquema: lo construyes desde los metadatos del escenario.

In [ ]:
# Explora el escenario: esto es lo único que te dan
import json, pathlib
base = pathlib.Path(".").resolve()
while not (base / "data_benchmark_causal").is_dir():
    base = base.parent
esc = base / "data_benchmark_causal" / "causal_benchmark_synthetic" / "datasets" / "real" / "ihdp"
meta = json.loads((esc / "metadata.json").read_text())
ident = json.loads((esc / "identification.json").read_text())
print("descripción:", meta["description"])
print("tratamiento:", meta["treatment"], "| resultado:", meta["outcome"])
print("covariables:", ident["valid_adjustment_sets"][0])
print("referencia del DGP:", meta["what_is_real"][:70])


In [ ]:
# Arranque de la tarea: el escenario se cambia con una variable de entorno
import os
os.environ["SIC_ESCENARIO"] = "real/ihdp"
print("ESCENARIO ->", os.environ["SIC_ESCENARIO"])
# Tu adaptación (edítalos en los .py y vuelve a correr los make):
#   datos.py      : EsquemaCausal -> tratamiento 'treatment', resultado 'outcome'
#                  y las 25 covariables que acabas de listar.
#   estimacion.py : estratos_nodegree -> TU corte, pre-declarado con su regla.
#   monitor.py   : ventana_deriva y ventana_positividad -> adáptalas a una
#                  covariable continua de ihdp (p. ej. bw o momage).
# Luego: make verificar && make estimar && make registrar && make ventanas && make humo


### Tus números (`real/ihdp`)

Completa la tabla con **tus** resultados y adjunta las salidas de los checkpoints:

| Pieza | Tu resultado |
|---|---|
| Diferencia cruda (train) | |
| ATE OLS + IC95 | |
| **ATE AIPW (cross-fitting) + IC95** | |
| ESS / n y rango de $\hat e$ | |
| Estrato 1 (corte que elegiste) + IC | |
| Estrato 2 + IC | |
| ¿Los intervalos de los estratos se solapan? | |
| Ventana sana → estado | |
| Ventana deriva → estado y señal | |
| Ventana positividad → estado y señal | |
| Humo: `/efecto` y `/consulta` | |

**Reglas de la tarea:**

- El corte de heterogeneidad es **tuyo**: decláralo *antes* de mirar los efectos (interpretabilidad o balance — nunca «porque salió la mayor diferencia»), y escríbelo en la tarjeta con la regla.
- Los **umbrales de la puerta** se recalibran para ihdp (¿el solapamiento mínimo de 0.50 sigue siendo razonable con 25 covariables y P(D=1) cercana a 0.43?): justifica tu elección en dos líneas.
- La referencia del DGP (~4,0) **no se mira para estimar ni decidir**: la pared se mantiene. Solo se compara al final, como en `real/jobs`.

## 8. Preguntas de cierre (vinculadas a la rúbrica del proyecto final)

1. **Estimando:** ¿qué cubre el intervalo de tu AIPW en ihdp — y qué NO cubre? ¿Qué responderías si te preguntan «¿cuánto mejora este niño en particular?»?
2. **Dos piernas:** tu outcome nuisance sigue siendo lineal. ¿Qué le pasaría a esa pierna con 25 covariables y formas no lineales, y por qué el plug-in te preocupa más que el AIPW?
3. **El corte:** ¿por qué la regla del corte va en la tarjeta y no en el código? Da un ejemplo de corte que ganarías por varianza pero no deberías declarar (en `real/jobs`: `black`).
4. **La puerta:** con tu ventana de deriva de ihdp activada, ¿qué acción de las cinco corresponde, y qué le dirías al equipo de producto que pregunta «¿reentrenamos ya?»?
5. **La pared:** nombra dos decisiones que pudiste tomar *solo porque* la verdad no estaba en el paquete.

In [ ]:
# Tus respuestas de cierre (2–4 líneas cada una)
cierre_1 = """
...
"""
cierre_2 = """
...
"""
cierre_3 = """
...
"""
cierre_4 = """
...
"""
cierre_5 = """
...
"""
print("respuestas listas para la entrega")


## Qué entregar

Todo el detalle está en [`ENTREGA.md`](ENTREGA.md). En síntesis: este notebook con salidas y respuestas, tu `artifacts/` de ihdp (con `manifest.json` y `champion`), la tabla de números completa y las adaptaciones declaradas.